<a href="https://colab.research.google.com/github/naheelomara2002-png/Growth_Rate/blob/main/growth_rate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -upgrade -pre xee
!pip install -U geemap

In [ ]:
import ee
import geemap

In [ ]:
ee.Authenticate()
ee.Initialize(project ='graphite-space-457522-v1')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
map = geemap.Map()
map

In [ ]:
point = map.draw_last_feature.geometry()

In [ ]:
roi = ee.FeatureCollection("FAO/GAUL/2015/level0").filterBounds(point).map(lambda feature: feature.simplify(100))

In [ ]:
map.addLayer(roi)

In [ ]:

population_2000 = ee.ImageCollection("CIESIN/GPWv411/GPW_Population_Density").select('population_density').filterBounds(roi).filterDate('2000-01-01', '2000-12-31').first().clip(roi)
population_2020 = ee.ImageCollection("CIESIN/GPWv411/GPW_Population_Density").select('population_density').filterBounds(roi).filterDate('2020-01-01', '2020-12-31').first().clip(roi)
population_2000 = population_2000.updateMask(population_2000.gt(5))

Growth_rate = population_2020.subtract(population_2000).divide(population_2000).multiply(100).rename('growth_pc')

In [ ]:
stats = Growth_rate.reduceRegion(
    reducer = ee.Reducer.percentile([5, 50, 95]),
    geometry = roi.geometry(),
    scale = 1000, maxPixels = 1e9)
print(stats.getInfo())

{'growth_pc_p5': 39.958658047956064, 'growth_pc_p50': 51.80349211984448, 'growth_pc_p95': 86.27702818676867}


In [ ]:
vis = {'min' : 40, 'max' : 86, 'palette':['#ffffb2', '#fed976', '#feb24c', '#fd8d3c', '#e31a1c', '#800026']}

In [ ]:
map.addLayer(Growth_rate , vis , 'Growth_rate')
map.add_colorbar(vis , label='Growth Rate (%)', layer_name='Growth_rate')